# 26 — Earth environmental expansion

Existing development sites only. Run All reads pinned sources and writes a new run. It performs no network acquisition and does not produce mission suitability scores.

## Pinned acquisitions and scope

In [ ]:
import hashlib
import json
import sys
from datetime import datetime, timedelta
from pathlib import Path

import nbformat
import numpy as np
import pandas as pd
from osmium.io import Reader
from scipy.io import netcdf_file
from shapely.geometry import box, shape

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'outline.md').exists())
sys.path.insert(0, str(ROOT / 'notebooks'))
sys.path.insert(0, str(ROOT / 'download scripts'))
from download_osm import checksums
from download_terraclimate import validate_netcdf
from osm_processing import extract_support
from processing import Run, decode_packed, sha256

BASE = ROOT / 'data/raw/earth/environment_expansion/20261030'
AREAS = {'death_valley': {'bbox': (-117.10, 36.0, -116.60, 36.5), 'download_id': 'region_01', 'extract': 'northern-california'},
         'wadi_rum': {'bbox': (35.2, 29.4, 35.65, 29.8), 'download_id': 'region_02', 'extract': 'jordan'}}
CLIMATE_MANIFEST = BASE / 'terraclimate/manifests/20261029T161008286307Z_run.json'
selection_path = BASE / 'osm/manifests/selection.json'
coverage_path = BASE / 'osm/coverage/index-v1.json'
climate_manifest = json.loads(CLIMATE_MANIFEST.read_text(encoding='utf-8'))
selection = json.loads(selection_path.read_text(encoding='utf-8'))
coverage = json.loads(coverage_path.read_text(encoding='utf-8'))
coverage_receipt = coverage_path.with_suffix('.json.receipt.json')
assert sha256(coverage_path) == json.loads(coverage_receipt.read_text())['sha256']
assert climate_manifest['status'] == 'complete'
expected = {(a['download_id'], year, variable) for a in AREAS.values()
            for year in range(2015, 2025) for variable in ['tmin', 'tmax', 'ppt', 'ws', 'srad']}
jobs = climate_manifest['jobs']
assert len(jobs) == len(expected) == 100
assert {(job['area'], job['year'], job['variable']) for job in jobs} == expected
aliases = {value['download_id']: name for name, value in AREAS.items()}
sources = {source['region']: source for source in selection['sources']}
assert set(sources) == {'northern-california', 'jordan'}
assert all(source['filename'].endswith('-261025.osm.pbf') for source in sources.values())
for name, spec in AREAS.items():
    features = [f for f in coverage['features'] if f['properties']['urls']['pbf'] == sources[spec['extract']]['latest_url']]
    assert len(features) == 1 and shape(features[0]['geometry']).covers(box(*spec['bbox'])), name
NOTEBOOK = 'preprocessing/26_earth_environment_expansion.ipynb'
notebook_path = ROOT / 'notebooks' / NOTEBOOK
notebook = nbformat.read(notebook_path, as_version=4)
code_hash = hashlib.sha256(json.dumps([cell.source for cell in notebook.cells if cell.cell_type == 'code']).encode()).hexdigest()
config = {'areas': AREAS, 'climate_period': '2015–2024', 'variables': ['tmin', 'tmax', 'ppt', 'ws', 'srad'],
          'climate_weighting': 'cos(latitude), native centers inside bbox',
          'osm_attribution': selection['attribution'], 'osm_license': selection['license'],
          'osm_polygon_epoch': 'current provider index; not proof of exact historical clipping geometry',
          'notebook_code_sha256': code_hash}
print('Validated 100 climate requests and current provider coverage for both dated OSM sources.')

## Decode native climate and extract support features

In [ ]:
with Run(ROOT, 'earth_environment_expansion', config, NOTEBOOK) as run:
    run.path('notebook_at_start.ipynb').write_bytes(notebook_path.read_bytes())
    for path in [CLIMATE_MANIFEST, selection_path, coverage_path, coverage_receipt,
                 ROOT / 'download scripts/download_terraclimate.py', ROOT / 'download scripts/download_osm.py',
                 ROOT / 'notebooks/osm_processing.py', ROOT / 'notebooks/diagnostics/environment-expansion-research-20261030/SOURCES.md']:
        run.input(path)
    summaries = []
    for job in jobs:
        area = aliases[job['area']]
        bounds = AREAS[area]['bbox']
        assert tuple(job['bounds']) == bounds
        source_path = BASE / 'terraclimate' / job['path']
        receipt_path = source_path.with_suffix('.nc.json')
        receipt = json.loads(receipt_path.read_text(encoding='utf-8'))
        assert receipt['source'] == job['source'] and sha256(source_path) == receipt['sha256']
        metadata = validate_netcdf(source_path, job['variable'], job['year'], bounds)
        run.input(source_path)
        run.input(receipt_path)
        with netcdf_file(str(source_path), 'r', mmap=False, maskandscale=False) as dataset:
            variable = dataset.variables[job['variable']]
            values = decode_packed(variable[:], [getattr(variable, '_FillValue', -32768), getattr(variable, 'missing_value', -32768)],
                                   float(getattr(variable, 'scale_factor', 1)), float(getattr(variable, 'add_offset', 0)))
            lon, lat = np.meshgrid(dataset.variables['lon'][:], dataset.variables['lat'][:])
            west, south, east, north = bounds
            inside = (lon >= west) & (lon < east) & (lat >= south) & (lat < north)
            weights = np.cos(np.deg2rad(lat))
            dates = [datetime(1900, 1, 1) + timedelta(days=float(t)) for t in dataset.variables['time'][:]]
            records = []
            for index, date in enumerate(dates):
                field = values[index]
                valid = inside & np.isfinite(field)
                assert valid.any()
                records.append(pd.DataFrame({'date': date.date().isoformat(), 'longitude': lon[inside], 'latitude': lat[inside],
                                             'value': field[inside], 'valid': valid[inside], 'variable': job['variable'], 'units': metadata['units']}))
                summaries.append({'area': area, 'date': date.date().isoformat(), 'variable': job['variable'], 'units': metadata['units'],
                                  'spatial_weighted_mean': float(np.average(field[valid], weights=weights[valid])),
                                  'spatial_min_of_monthly_field': float(field[valid].min()), 'spatial_max_of_monthly_field': float(field[valid].max()),
                                  'valid_cells': int(valid.sum()), 'selected_cells': int(inside.sum()),
                                  'weighted_coverage': float(weights[valid].sum() / weights[inside].sum())})
            pd.concat(records, ignore_index=True).to_csv(run.path(f"earth/{area}/native_monthly/{job['variable']}_{job['year']}.csv.gz"),
                                                       index=False, compression={'method': 'gzip', 'mtime': 0})
    monthly = pd.DataFrame(summaries)
    assert len(monthly) == 1200 and not monthly.duplicated(['area', 'date', 'variable']).any()
    monthly.to_csv(run.path('regional_monthly_summary.csv'), index=False)
    print(monthly.groupby('area').agg(monthly_fields=('variable', 'size'), minimum_coverage=('weighted_coverage', 'min')))
    inventory = []
    for area, spec in AREAS.items():
        source = sources[spec['extract']]
        path = BASE / 'osm/extracts' / source['filename']
        receipt_path = path.with_suffix(path.suffix + '.json')
        receipt = json.loads(receipt_path.read_text(encoding='utf-8'))
        md5, digest = checksums(path)
        assert receipt['source'] == source and path.stat().st_size == source['size_bytes']
        assert md5 == source['md5'] and digest == receipt['sha256']
        run.input(path)
        run.input(receipt_path)
        with Reader(str(path)) as reader:
            header = reader.header()
            timestamp = header.get('osmosis_replication_timestamp', '')
            replication_sequence = header.get('osmosis_replication_sequence_number', '')
            generator = header.get('generator', '')
        quality = extract_support([path], spec['bbox'], run.path(f'earth/{area}/support_features.geojson'))
        quality.update({'source_filename': source['filename'], 'source_replication_timestamp': timestamp or 'unavailable',
                        'attribution': selection['attribution'], 'license': selection['license']})
        run.path(f'earth/{area}/quality.json').write_text(json.dumps(quality, indent=2), encoding='utf-8')
        inventory.append({'area': area, 'source': source, 'replication_timestamp': timestamp,
                          'replication_sequence_number': replication_sequence, 'generator': generator,
                          'current_provider_polygon_covers_box': True})
        print(area, quality)
    run.path('source_inventory.json').write_text(json.dumps(inventory, indent=2), encoding='utf-8')
    run.path('interpretation.json').write_text(json.dumps({
        'climate': 'native 1/24 degree monthly estimates; no daily extremes, forecasts or 30m resolution claim',
        'support': 'clipped existing OSM selection rules; absent/incomplete mappings and access remain Unknown',
        'scope': 'Death Valley and Wadi Rum development boxes only',
        'coverage': 'current provider index covers both rectangles; historical clipping geometry is not certified',
        'snapshot': 'starting notebook is archived because execution outputs change the final notebook bytes'}, indent=2), encoding='utf-8')
print('Completed:', run.directory)